# Calibration dry-run

Run this **once before class** to verify the game is tuned correctly. Simulates all four team archetypes (OFAT, fractional with center points, fractional without center points, CCD) playing round 1 and round 2, then prints a verdict on whether the pedagogical math still plays.

**All 7 verdict checks must PASS** or the game outcome won't land as intended. If any FAIL, edit the coefficients in the true-model cell (below) and rerun.

**Run in Colab:** this notebook is self-contained — no local files or uploads needed. Just open it in Colab and Runtime → Run all. The only non-stdlib dependency is `numpy`, which Colab already has; the `%pip install` line in the second code cell is there in case you're running somewhere that doesn't.

## What the verdict checks

1. **CCD_beats_OFAT_on_elongation** — CCD's round-2 elongation is ≥ 30% higher than OFAT's.
2. **CCD_tensile_customer_ok** — CCD's tensile at its Pareto pick stays above the 6 MPa floor.
3. **fracC_detects_curvature** — fractional-with-center-points detects curvature via lack-of-fit.
4. **fracN_cannot_detect** — fractional-without-center-points cannot detect curvature.
5. **fracC_beats_OFAT_on_elong** — with center points, fractional team beats OFAT on elongation.
6. **tensile_SNR>3** — main-effect range at ±1 corners exceeds 3× noise SD.
7. **elongation_curvature_empirical_SNR>3** — empirical curvature detection SNR exceeds 3.

## Setup — the true model

Same as `true-model.ipynb`. This cell defines the physics.

In [ ]:
"""Latex rubber film formulation — physics simulator for the DOE tutorial game.

Public API: `simulate(**natural_units) -> dict`.
"""
from __future__ import annotations
import math
from dataclasses import dataclass


@dataclass(frozen=True)
class FactorRange:
    name: str
    low: float
    high: float

    def to_coded(self, x: float) -> float:
        mid = 0.5 * (self.low + self.high)
        half = 0.5 * (self.high - self.low)
        return (x - mid) / half

    def from_coded(self, x_c: float) -> float:
        mid = 0.5 * (self.low + self.high)
        half = 0.5 * (self.high - self.low)
        return mid + x_c * half


RANGES = {
    "latex_pct":  FactorRange("latex_pct",  5.0,  20.0),
    "filler_phr":        FactorRange("filler_phr",        0.0,  40.0),
    "crosslinker_phr":   FactorRange("crosslinker_phr",   0.5,   4.0),
    "plasticizer_phr":   FactorRange("plasticizer_phr",   0.0,  25.0),
    "cure_temp_c":       FactorRange("cure_temp_c",     100.0, 160.0),
}

LATEX_HARD_CAP = 20.0

TENSILE = dict(
    intercept  = 8.0,
    A          = 0.10,
    B          = 3.00,
    C          = 2.00,
    D          = -0.20,
    E          = 0.20,
    BC         = 1.50,
    noise_sd   = 0.60,
)

ELONGATION = dict(
    intercept       = 380.0,
    A_linear        =   3.0,
    E_linear        =   8.0,
    B_center        =  -0.5,
    B_curvature     = -100.0,
    C_center        =  +0.5,
    C_curvature     = -150.0,
    D_center        =  +0.5,
    D_curvature     =  -80.0,
    BC              =  -10.0,
    noise_sd        =  15.0,
)

HARDNESS = dict(
    intercept  = 55.0,
    A          =  0.5,
    B          = 12.0,
    C          =  5.0,
    D          = -3.0,
    E          =  2.0,
    BC         =  3.0,
    noise_sd   =  2.5,
)


def _tensile(A, B, C, D, E, rng):
    m = TENSILE
    y = (m["intercept"] + m["A"]*A + m["B"]*B + m["C"]*C + m["D"]*D + m["E"]*E
         + m["BC"]*B*C)
    return y + rng.gauss(0.0, m["noise_sd"])


def _elongation(A, B, C, D, E, rng):
    m = ELONGATION
    y = (m["intercept"]
         + m["A_linear"]*A
         + m["E_linear"]*E
         + m["B_curvature"]*(B - m["B_center"])**2
         + m["C_curvature"]*(C - m["C_center"])**2
         + m["D_curvature"]*(D - m["D_center"])**2
         + m["BC"]*B*C)
    return y + rng.gauss(0.0, m["noise_sd"])


def _hardness(A, B, C, D, E, rng):
    m = HARDNESS
    y = (m["intercept"] + m["A"]*A + m["B"]*B + m["C"]*C + m["D"]*D + m["E"]*E
         + m["BC"]*B*C)
    return y + rng.gauss(0.0, m["noise_sd"])


def _in_range(name, x):
    r = RANGES[name]
    if x < r.low or x > r.high:
        return f"{name}={x} outside [{r.low}, {r.high}]"
    return None


def simulate(latex_pct, filler_phr, crosslinker_phr,
             plasticizer_phr, cure_temp_c, *, rng=None):
    """Return a dict with the three responses plus feasibility flag."""
    import random
    if rng is None:
        rng = random.Random()

    for nm, x in (("latex_pct", latex_pct),
                  ("filler_phr", filler_phr),
                  ("crosslinker_phr", crosslinker_phr),
                  ("plasticizer_phr", plasticizer_phr),
                  ("cure_temp_c", cure_temp_c)):
        problem = _in_range(nm, x)
        if problem:
            return dict(tensile_mpa=math.nan, elongation_pct=math.nan,
                        hardness_shore_a=math.nan, infeasible=problem)

    if latex_pct > LATEX_HARD_CAP:
        return dict(tensile_mpa=math.nan, elongation_pct=math.nan,
                    hardness_shore_a=math.nan,
                    infeasible=f"latex > {LATEX_HARD_CAP}% — mixture unstable")

    A = RANGES["latex_pct"].to_coded(latex_pct)
    B = RANGES["filler_phr"].to_coded(filler_phr)
    C = RANGES["crosslinker_phr"].to_coded(crosslinker_phr)
    D = RANGES["plasticizer_phr"].to_coded(plasticizer_phr)
    E = RANGES["cure_temp_c"].to_coded(cure_temp_c)

    return dict(
        tensile_mpa=_tensile(A, B, C, D, E, rng),
        elongation_pct=_elongation(A, B, C, D, E, rng),
        hardness_shore_a=_hardness(A, B, C, D, E, rng),
        infeasible=None,
    )


def simulate_coded(A, B, C, D, E, *, rng=None):
    """Coded [-1,+1] factor levels -> responses."""
    return simulate(
        RANGES["latex_pct"].from_coded(A),
        RANGES["filler_phr"].from_coded(B),
        RANGES["crosslinker_phr"].from_coded(C),
        RANGES["plasticizer_phr"].from_coded(D),
        RANGES["cure_temp_c"].from_coded(E),
        rng=rng,
    )


## Design generators + analysis routines

Each team archetype has a design generator and an analysis fitting routine. These match what students actually do — OFAT teams pick per-factor bests and combine; fractional teams fit 2-level linear + interactions; CCD teams fit a full quadratic.

In [ ]:
# %pip install numpy

import random
import math
import itertools
import statistics
import numpy as np

SEED = 20260819  # bootcamp date
BUDGET_ROUND1 = 15
BUDGET_ROUND2 = 5


def ofat_design():
    """5 factors × 3 levels = 15 runs, one factor swept at a time."""
    runs = []
    for f in "ABCDE":
        for lvl in (-1.0, 0.0, 1.0):
            row = {letter: 0.0 for letter in "ABCDE"}
            row[f] = lvl
            runs.append(row)
    return runs


def fracfact_design_no_center():
    """2^(4-1) + fold-over + 3 interior probes = 15."""
    corners = list(itertools.product([-1.0, 1.0], repeat=3))
    runs = []
    for a, b, c in corners:
        d = a * b * c
        runs.append({"A": a, "B": b, "C": c, "D": d, "E": 0.0})
    for a, b, c in corners[:4]:
        d = -(a * b * c)
        runs.append({"A": -a, "B": -b, "C": -c, "D": d, "E": 0.0})
    for f in ["B", "C", "D"]:
        row = {letter: 0.0 for letter in "ABCDE"}
        row[f] = 0.5
        runs.append(row)
    return runs


def fracfact_design_with_center():
    """Same 2^(4-1) but replace the 3 interior probes with 3 center-point runs."""
    corners = list(itertools.product([-1.0, 1.0], repeat=3))
    runs = []
    for a, b, c in corners:
        d = a * b * c
        runs.append({"A": a, "B": b, "C": c, "D": d, "E": 0.0})
    for a, b, c in corners[:4]:
        d = -(a * b * c)
        runs.append({"A": -a, "B": -b, "C": -c, "D": d, "E": 0.0})
    for _ in range(3):
        runs.append({letter: 0.0 for letter in "ABCDE"})
    return runs


def ccd_design():
    """Face-centered CCD on B,C,D: 8 corners + 6 axial + 1 center = 15."""
    corners = list(itertools.product([-1.0, 1.0], repeat=3))
    runs = []
    for b, c, d in corners:
        runs.append({"A": 0.0, "B": b, "C": c, "D": d, "E": 0.0})
    for f, sign in [("B", -1), ("B", +1), ("C", -1), ("C", +1), ("D", -1), ("D", +1)]:
        row = {letter: 0.0 for letter in "ABCDE"}
        row[f] = float(sign)
        runs.append(row)
    runs.append({letter: 0.0 for letter in "ABCDE"})
    return runs


def run_design(runs, rng):
    out = []
    for row in runs:
        result = simulate_coded(row["A"], row["B"], row["C"], row["D"], row["E"], rng=rng)
        merged = dict(row)
        merged.update(result)
        out.append(merged)
    return out

In [ ]:
def fit_ofat_and_predict_best(rows, response_key):
    """OFAT team: per-factor best level, combined greedily."""
    best_per_factor = {}
    for f in "ABCDE":
        f_rows = [row for row in rows
                  if all(row[x] == 0.0 for x in "ABCDE" if x != f)]
        if not f_rows:
            best_per_factor[f] = 0.0
            continue
        best = max(f_rows, key=lambda r: r[response_key] if not math.isnan(r[response_key]) else -1e9)
        best_per_factor[f] = best[f]
    return best_per_factor


def fit_2level_linear_and_predict(rows, response_key):
    """Main effects + all 2-way interactions on corners."""
    corners = [r for r in rows if all(abs(r[k]) == 1.0 for k in "ABCD")]
    if not corners:
        return {}, None, None
    terms = [
        ("1", lambda r: 1.0),
        ("A", lambda r: r["A"]), ("B", lambda r: r["B"]),
        ("C", lambda r: r["C"]), ("D", lambda r: r["D"]),
        ("AB", lambda r: r["A"]*r["B"]),
        ("AC", lambda r: r["A"]*r["C"]),
        ("AD", lambda r: r["A"]*r["D"]),
        ("BC", lambda r: r["B"]*r["C"]),
        ("BD", lambda r: r["B"]*r["D"]),
        ("CD", lambda r: r["C"]*r["D"]),
    ]
    n = len(corners)
    if n < len(terms):
        terms = terms[:min(n, len(terms))]
    X = np.array([[fn(r) for _, fn in terms] for r in corners])
    y = np.array([r[response_key] for r in corners])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    coefs = dict(zip((name for name, _ in terms), coef))

    grid = list(itertools.product([-1.0, -0.5, 0.0, 0.5, 1.0], repeat=4))
    best_y, best_pt = -1e9, None
    for a, b, c, d in grid:
        row = {"A": a, "B": b, "C": c, "D": d, "E": 0.0}
        y_hat = sum(fn(row) * coefs.get(name, 0.0) for name, fn in terms)
        if y_hat > best_y:
            best_y, best_pt = y_hat, row
    return coefs, best_pt, best_y


def lof_center_test(rows, response_key):
    """Center-point lack-of-fit test."""
    corners = [r for r in rows if all(abs(r[k]) == 1.0 for k in "ABCD")]
    centers = [r for r in rows if all(r[k] == 0.0 for k in "ABCDE")]
    if not corners or len(centers) < 2:
        return None
    mc = statistics.mean([r[response_key] for r in corners])
    m0 = statistics.mean([r[response_key] for r in centers])
    sd0 = statistics.stdev([r[response_key] for r in centers])
    curvature = m0 - mc
    detected = abs(curvature) > 2 * (sd0 if sd0 > 0 else 1e-9)
    return dict(mean_corner=mc, mean_center=m0, sd_center=sd0,
                curvature=curvature, detected=detected)


def fit_ccd_quadratic_and_predict(rows, response_key):
    """Full quadratic in B, C, D over all 15 runs."""
    terms = [
        ("1", lambda r: 1.0),
        ("B", lambda r: r["B"]), ("C", lambda r: r["C"]), ("D", lambda r: r["D"]),
        ("BC", lambda r: r["B"]*r["C"]),
        ("BD", lambda r: r["B"]*r["D"]),
        ("CD", lambda r: r["C"]*r["D"]),
        ("BB", lambda r: r["B"]**2),
        ("CC", lambda r: r["C"]**2),
        ("DD", lambda r: r["D"]**2),
    ]
    X = np.array([[fn(r) for _, fn in terms] for r in rows])
    y = np.array([r[response_key] for r in rows])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    coefs = dict(zip((name for name, _ in terms), coef))
    grid = list(itertools.product(np.linspace(-1, 1, 11), repeat=3))
    best_y, best_pt = -1e9, None
    for b, c, d in grid:
        row = {"A": 0.0, "B": b, "C": c, "D": d, "E": 0.0}
        y_hat = sum(fn(row) * coefs.get(name, 0.0) for name, fn in terms)
        if y_hat > best_y:
            best_y, best_pt = y_hat, row
    return coefs, best_pt, best_y


def evaluate_true_at(pt, rng, reps=5):
    ys = [simulate_coded(pt["A"], pt["B"], pt["C"], pt["D"], pt["E"], rng=rng)
          for _ in range(reps)]
    return dict(
        tensile_mean=statistics.mean(y["tensile_mpa"] for y in ys),
        tensile_sd=statistics.stdev(y["tensile_mpa"] for y in ys),
        elongation_mean=statistics.mean(y["elongation_pct"] for y in ys),
        elongation_sd=statistics.stdev(y["elongation_pct"] for y in ys),
    )

## Run the calibration

Executes each team's round-1 design against the true model, has each team fit the model their design supports, then simulates round 2 and produces the verdict.

In [ ]:
print("=" * 78)
print("LATEX FORMULATION TRUE-MODEL — CALIBRATION DRY RUN")
print("=" * 78)

rng = random.Random(SEED)

ofat_runs   = run_design(ofat_design(), rng)
fracC_runs  = run_design(fracfact_design_with_center(), rng)
fracN_runs  = run_design(fracfact_design_no_center(), rng)
ccd_runs    = run_design(ccd_design(), rng)

print("\n" + "-" * 78)
print("ROUND 1: teams optimize tensile from 15 runs")
print("-" * 78)

ofat_best_pt = fit_ofat_and_predict_best(ofat_runs, "tensile_mpa")
ofat_best_pt["E"] = ofat_best_pt.get("E", 0.0)
fracC_coefs, fracC_best_pt, fracC_best_yhat = fit_2level_linear_and_predict(fracC_runs, "tensile_mpa")
fracN_coefs, fracN_best_pt, fracN_best_yhat = fit_2level_linear_and_predict(fracN_runs, "tensile_mpa")
ccd_coefs, ccd_best_pt, ccd_best_yhat = fit_ccd_quadratic_and_predict(ccd_runs, "tensile_mpa")

print(f"\nOFAT predicted-best tensile: {ofat_best_pt}")
print(f"Frac (with center) predicted-best: {fracC_best_pt} -> yhat={fracC_best_yhat:.2f}")
print(f"Frac (no center) predicted-best:   {fracN_best_pt} -> yhat={fracN_best_yhat:.2f}")
print(f"CCD predicted-best:                {ccd_best_pt} -> yhat={ccd_best_yhat:.2f}")

truth_ofat  = evaluate_true_at(ofat_best_pt, random.Random(SEED + 1))
truth_fracC = evaluate_true_at(fracC_best_pt, random.Random(SEED + 2))
truth_fracN = evaluate_true_at(fracN_best_pt, random.Random(SEED + 3))
truth_ccd   = evaluate_true_at(ccd_best_pt, random.Random(SEED + 4))

print("\nActual tensile at each team's predicted-best (mean over 5 reps):")
print(f"  OFAT       : {truth_ofat['tensile_mean']:6.2f} ± {truth_ofat['tensile_sd']:.2f}")
print(f"  Frac (C)   : {truth_fracC['tensile_mean']:6.2f} ± {truth_fracC['tensile_sd']:.2f}")
print(f"  Frac (N)   : {truth_fracN['tensile_mean']:6.2f} ± {truth_fracN['tensile_sd']:.2f}")
print(f"  CCD        : {truth_ccd['tensile_mean']:6.2f} ± {truth_ccd['tensile_sd']:.2f}")

In [ ]:
print("-" * 78)
print("ROUND 2: customer twist — also maximize elongation. +5 experiment-$")
print("-" * 78)

ofat_round2 = [{**ofat_best_pt, "A": ofat_best_pt.get("A", 0.0), "E": 0.0}] * 5
ofat_all = ofat_runs + run_design(ofat_round2, rng)

fracC_round2 = [
    {"A":0.0,"B":-1.0,"C":0.0,"D":0.0,"E":0.0},
    {"A":0.0,"B":+1.0,"C":0.0,"D":0.0,"E":0.0},
    {"A":0.0,"B":0.0,"C":-1.0,"D":0.0,"E":0.0},
    {"A":0.0,"B":0.0,"C":+1.0,"D":0.0,"E":0.0},
    {"A":0.0,"B":0.0,"C":0.0,"D":0.0,"E":0.0},
]
fracC_all = fracC_runs + run_design(fracC_round2, rng)
fracN_all = fracN_runs + run_design(fracC_round2, rng)

ccd_pareto_pt = fit_ccd_quadratic_and_predict(ccd_runs, "elongation_pct")[1]
ccd_all = ccd_runs + run_design([ccd_pareto_pt] * 5, rng)

lof_fracC = lof_center_test(fracC_all, "elongation_pct")
lof_fracN = lof_center_test(fracN_all, "elongation_pct")
lof_ccd   = lof_center_test(ccd_all, "elongation_pct")

print("\nElongation curvature detection (center-point LOF):")
for tag, lof in [("Frac (with ctr)", lof_fracC),
                 ("Frac (no ctr)",   lof_fracN),
                 ("CCD",             lof_ccd)]:
    if lof is None:
        print(f"  {tag:20}: no centers (undetectable)")
    else:
        print(f"  {tag:20}: curvature={lof['curvature']:+6.1f}  detected={lof['detected']}")

ofat_elong_best_pt   = fit_ofat_and_predict_best(ofat_all, "elongation_pct")
_, fracC_elong_best_pt, _ = fit_ccd_quadratic_and_predict(fracC_all, "elongation_pct")
_, fracN_elong_best_pt, _ = fit_2level_linear_and_predict(fracN_all, "elongation_pct")
_, ccd_pareto_pt, _  = fit_ccd_quadratic_and_predict(ccd_all, "elongation_pct")

truth_ofat_2  = evaluate_true_at(ofat_elong_best_pt,  random.Random(SEED + 11))
truth_fracC_2 = evaluate_true_at(fracC_elong_best_pt, random.Random(SEED + 12))
truth_fracN_2 = evaluate_true_at(fracN_elong_best_pt, random.Random(SEED + 13))
truth_ccd_2   = evaluate_true_at(ccd_pareto_pt,       random.Random(SEED + 14))

print("\n(tensile, elongation) at each team's round-2 best-elong pick:")
for tag, t in [("OFAT",          truth_ofat_2),
               ("Frac (w/ ctr)", truth_fracC_2),
               ("Frac (no ctr)", truth_fracN_2),
               ("CCD",           truth_ccd_2)]:
    print(f"  {tag:15}: tensile={t['tensile_mean']:6.2f}  elongation={t['elongation_mean']:6.1f}")

In [ ]:
print("\n" + "=" * 78)
print("VERDICT")
print("=" * 78)

verdict = {}
TENSILE_FLOOR = 6.0
ELONG_GAP = 30.0

verdict["CCD_beats_OFAT_on_elongation"] = (
    truth_ccd_2["elongation_mean"] - truth_ofat_2["elongation_mean"] > ELONG_GAP
)
verdict["CCD_tensile_customer_ok"] = truth_ccd_2["tensile_mean"] >= TENSILE_FLOOR
verdict["fracC_detects_curvature"] = bool(lof_fracC and lof_fracC["detected"])
verdict["fracN_cannot_detect"]     = (lof_fracN is None) or (not lof_fracN["detected"])
verdict["fracC_beats_OFAT_on_elong"] = (
    truth_fracC_2["elongation_mean"] > truth_ofat_2["elongation_mean"]
)
tensile_corner_range = 2 * TENSILE["B"] + 2 * TENSILE["C"]
verdict["tensile_SNR>3"] = (tensile_corner_range / TENSILE["noise_sd"]) > 3
if lof_fracC:
    empirical_snr = abs(lof_fracC["curvature"]) / max(lof_fracC["sd_center"], 1e-6)
    verdict["elongation_curvature_empirical_SNR>3"] = empirical_snr > 3
else:
    verdict["elongation_curvature_empirical_SNR>3"] = False

for k, v in verdict.items():
    mark = "PASS" if v else "FAIL"
    print(f"  [{mark}] {k}: {v}")

all_pass = all(verdict.values())
print()
print("OVERALL: " + ("PASS ✓" if all_pass else "FAIL ✗ — tune coefficients"))